# Reconnaissance de lieux de Terre Sainte

Transfer learning : ResNet18 (pré-entraîné sur ImageNet) sert d'extracteur de features,
puis un classifieur simple est entraîné sur ces features.

Structure attendue sur Google Drive (mêmes sous-dossiers dans `train` et `test`) :
```
MyDrive/lieux/
  train/<lieu>/*.jpg
  test/<lieu>/*.jpg
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/lieux'

## Extraction des features

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# prétraitement standard ImageNet
tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def extraire_features(model, dossier):
    ds = datasets.ImageFolder(dossier, transform=tf)
    loader = DataLoader(ds, batch_size=32, shuffle=False)
    model.eval().to(device)
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            feats.append(model(x.to(device)).cpu().numpy())
            labels.append(y.numpy())
    chemins = [c for c, _ in ds.samples]
    return np.concatenate(feats), np.concatenate(labels), ds.classes, chemins

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()  # on garde le vecteur de 512 features

X_train, y_train, classes, _ = extraire_features(resnet, f'{DATA}/train')
X_test, y_test, classes_test, chemins_test = extraire_features(resnet, f'{DATA}/test')

assert classes == classes_test
print('Classes :', classes)
print('Train :', X_train.shape, ' Test :', X_test.shape)

## Classifieurs

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

classifieurs = {
    'Régression logistique': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'k-NN (k=3)': KNeighborsClassifier(n_neighbors=3, metric='cosine'),
}

resultats = {}
for nom, clf in classifieurs.items():
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    resultats[f'Pré-entraîné + {nom}'] = acc
    print(f'{nom} : {acc:.1%}')

## Matrice de confusion

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# régression logistique choisie à l'avance (pas sur la base du test)
logreg = classifieurs['Régression logistique']
pred = logreg.predict(X_test)

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=classes, xticks_rotation=45)
plt.title('Matrice de confusion (test)')
plt.tight_layout()
plt.show()

print('Erreurs :')
for chemin, vrai, p in zip(chemins_test, y_test, pred):
    if vrai != p:
        print(f'  {chemin.split("/")[-1]} : vrai = {classes[vrai]}, prédit = {classes[p]}')

## Sans pré-entraînement

Même architecture avec des poids aléatoires, pour mesurer l'apport d'ImageNet.

In [ ]:
torch.manual_seed(0)
resnet_alea = models.resnet18(weights=None)
resnet_alea.fc = torch.nn.Identity()

X0_train, y0_train, _, _ = extraire_features(resnet_alea, f'{DATA}/train')
X0_test, y0_test, _, _ = extraire_features(resnet_alea, f'{DATA}/test')

clf0 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
clf0.fit(X0_train, y0_train)
resultats['Sans pré-entraînement + Régression logistique'] = accuracy_score(y0_test, clf0.predict(X0_test))

print(f'Hasard : {1/len(classes):.1%}')
for nom, acc in resultats.items():
    print(f'{nom} : {acc:.1%}')

## Export ONNX (pour la démo web)

Le StandardScaler et la régression logistique sont des opérations linéaires :
on les fusionne en une couche `Linear` ajoutée à la fin de ResNet18,
puis on exporte le tout en un seul fichier ONNX.

In [ ]:
!pip install -q onnx onnxruntime
import onnxruntime as ort

scaler = logreg.named_steps['standardscaler']
lr = logreg.named_steps['logisticregression']
W = lr.coef_ / scaler.scale_
b = lr.intercept_ - (lr.coef_ * scaler.mean_ / scaler.scale_).sum(axis=1)

modele = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
modele.fc = torch.nn.Linear(512, len(classes))
modele.fc.weight.data = torch.tensor(W, dtype=torch.float32)
modele.fc.bias.data = torch.tensor(b, dtype=torch.float32)
modele = torch.nn.Sequential(modele, torch.nn.Softmax(dim=1)).eval()

# vérification : mêmes probabilités que scikit-learn
ds_test = datasets.ImageFolder(f'{DATA}/test', transform=tf)
x = torch.stack([ds_test[i][0] for i in range(len(ds_test))])
with torch.no_grad():
    p_torch = modele(x).numpy()
print('Écart avec scikit-learn :', np.abs(p_torch - logreg.predict_proba(X_test)).max())

chemin = f'{DATA}/modele.onnx'
torch.onnx.export(modele, torch.randn(1, 3, 224, 224), chemin,
                  input_names=['image'], output_names=['proba'], opset_version=17, dynamo=False)
p_onnx = ort.InferenceSession(chemin).run(None, {'image': x[:1].numpy()})[0]
print('Écart ONNX / PyTorch :', np.abs(p_onnx - p_torch[:1]).max())